In [1]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para visualización (las usaremos en el EDA)
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/madrid_airbnb - madrid_airbnb.csv')
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,6369,"Rooftop terrace room , ensuite bathroom",13660,Simon,Chamartín,Hispanoamérica,4.045.724,-367.688,Private room,60,1,78,2020-09-20,0.58,1,180
1,21853,Bright and airy room,83531,Abdel,Latina,Cármenes,4.040.381,-37.413,Private room,31,4,33,2018-07-15,0.42,2,364
2,23001,Apartmento Arganzuela- Madrid Rio,82175,Jesus,Arganzuela,Legazpi,403.884,-369.511,Entire home/apt,50,15,0,NaN,NaN,7,1
3,24805,Gran Via Studio Madrid,346366726,A,Centro,Universidad,4.042.183,-370.529,Entire home/apt,92,5,10,2020-03-01,0.13,1,72
4,26825,Single Room whith private Bathroom,114340,Agustina,Arganzuela,Legazpi,4.038.975,-369.018,Private room,26,2,149,2020-03-12,1.12,1,365


In [2]:
# Dimensiones del dataset: (filas, columnas)
print(df.shape)

# Información general: tipos de datos y valores no nulos por columna
df.info()

(19618, 16)
<class 'pandas.DataFrame'>
RangeIndex: 19618 entries, 0 to 19617
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              19618 non-null  int64  
 1   name                            19615 non-null  str    
 2   host_id                         19618 non-null  int64  
 3   host_name                       19091 non-null  str    
 4   neighbourhood_group             19618 non-null  str    
 5   neighbourhood                   19618 non-null  str    
 6   latitude                        19618 non-null  str    
 7   longitude                       19618 non-null  float64
 8   room_type                       19618 non-null  str    
 9   price                           19618 non-null  int64  
 10  minimum_nights                  19618 non-null  int64  
 11  number_of_reviews               19618 non-null  int64  
 12  last_review                    

In [3]:
# Ver valores únicos de latitude que no se puedan convertir a número
df[pd.to_numeric(df['latitude'], errors='coerce').isna()]['latitude'].unique()

<StringArray>
['4.045.724', '4.040.381', '4.042.183', '4.038.975', '4.041.476', '4.041.259',
 '4.041.844', '4.041.969', '4.042.247', '4.042.792',
 ...
 '4.045.236', '4.045.739', '4.043.562', '4.035.071', '4.043.019', '4.044.004',
 '4.044.046', '4.045.424', '4.046.203', '4.037.898']
Length: 6786, dtype: str

Se detectó que `latitude` tenía formato de texto con puntos mal ubicados. Se corrigió a tipo float reconstruyendo el valor decimal (ver celdas siguientes).


In [4]:
# Quitar los puntos y ver cuántos dígitos tiene cada valor de latitude
digitos = df['latitude'].str.replace('.', '', regex=False)
digitos.str.len().value_counts()

latitude
7    17644
6     1761
5      196
4       17
Name: count, dtype: int64

In [5]:
# comprobar que todos los valores empiezan por "40"
digitos.str[:2].value_counts()


latitude
40    19618
Name: count, dtype: int64

In [6]:
# Reconstruir latitude: quitar puntos, convertir a número y recolocar el decimal tras los 2 primeros dígitos
digitos = df['latitude'].str.replace('.', '', regex=False)
df['latitude'] = digitos.astype(float) / (10 ** (digitos.str.len() - 2))

# Comprobar el resultado
df['latitude'].head(10)
df['latitude'].dtype

dtype('float64')

In [7]:
# Ver ejemplos de longitude (ya es float, pero con el punto mal puesto)
df['longitude'].head(10)
df['longitude'].describe()

count    19618.000000
mean      -335.646818
std        101.079748
min       -386.391000
25%       -370.691000
50%       -369.973000
75%       -367.294250
max         -3.584000
Name: longitude, dtype: float64

In [8]:
# Quitar el signo y el punto, ver cuántos dígitos tiene cada valor
digitos_lon = df['longitude'].astype(str).str.replace('-', '', regex=False).str.replace('.', '', regex=False)
digitos_lon.str.len().value_counts()

# Comprobar que todos empiezan por "3" (parte entera de la longitud de Madrid)
digitos_lon.str[:1].value_counts()

longitude
3    19618
Name: count, dtype: int64

In [9]:
# Reconstruir longitude: quitar signo y punto, convertir a número, recolocar el decimal tras 1 dígito, y devolver el signo negativo
digitos_lon = df['longitude'].astype(str).str.replace('-', '', regex=False).str.replace('.', '', regex=False)
df['longitude'] = -(digitos_lon.astype(float) / (10 ** (digitos_lon.str.len() - 1)))

# Comprobar el resultado
print(df['longitude'].head(10))
print(df['longitude'].describe())

0   -3.67688
1   -3.74130
2   -3.69511
3   -3.70529
4   -3.69018
5   -3.69492
6   -3.70418
7   -3.70105
8   -3.71073
9   -3.69736
Name: longitude, dtype: float64
count    19618.000000
mean        -3.694040
std          0.028671
min         -3.863910
25%         -3.707700
50%         -3.701120
75%         -3.685420
max         -3.531900
Name: longitude, dtype: float64


In [10]:
# Verificar que latitude quedó correctamente reconstruida
df['latitude'].head(10)

0    40.45724
1    40.40381
2    40.38840
3    40.42183
4    40.38975
5    40.38860
6    40.41476
7    40.41259
8    40.41844
9    40.41969
Name: latitude, dtype: float64

In [11]:
# Ver nulos por columna (solo las que tienen alguno)
df.isnull().sum()[df.isnull().sum() > 0]

name                    3
host_name             527
last_review          5637
reviews_per_month    5637
dtype: int64

In [12]:
# Rellenar nulos según el criterio decidido
df['name'] = df['name'].fillna('Sin título')
df['host_name'] = df['host_name'].fillna('Desconocido')
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# Comprobar que ya no quedan nulos en estas tres columnas
df[['name', 'host_name', 'reviews_per_month']].isnull().sum()

name                 0
host_name            0
reviews_per_month    0
dtype: int64

In [13]:
# Comprobar si hay filas completamente duplicadas
df.duplicated().sum()

np.int64(0)

In [14]:
# Ver categorías únicas de room_type
print(df['room_type'].value_counts())
print()
# Ver categorías únicas de neighbourhood_group
print(df['neighbourhood_group'].value_counts())

room_type
Entire home/apt    11314
Private room        7809
Shared room          329
Hotel room           166
Name: count, dtype: int64

neighbourhood_group
Centro                   8649
Salamanca                1324
Chamberí                 1252
Arganzuela               1104
Tetuán                    816
Carabanchel               708
Retiro                    664
Ciudad Lineal             649
Puente de Vallecas        617
Latina                    608
Chamartín                 580
Moncloa - Aravaca         554
San Blas - Canillejas     497
Hortaleza                 364
Fuencarral - El Pardo     315
Usera                     280
Villaverde                177
Barajas                   172
Moratalaz                 118
Villa de Vallecas          98
Vicálvaro                  72
Name: count, dtype: int64


In [15]:
# Estadísticas descriptivas de price
df['price'].describe()

count    19618.000000
mean       129.271740
std        484.143545
min          0.000000
25%         35.000000
50%         58.000000
75%        100.000000
max       9999.000000
Name: price, dtype: float64

In [16]:
# Contar cuántos anuncios tienen price = 0
print('Anuncios con precio 0:', (df['price'] == 0).sum())

# Contar cuántos anuncios tienen precio muy alto (por ejemplo, más de 1000€/noche)
print('Anuncios con precio > 1000:', (df['price'] > 1000).sum())

# Ver los 10 precios más altos
print(df['price'].sort_values(ascending=False).head(10))

Anuncios con precio 0: 8
Anuncios con precio > 1000: 228
4056    9999
412     9999
2927    9999
3704    9999
4043    9999
2544    9999
4711    9999
9100    9999
9863    9999
4088    9999
Name: price, dtype: int64


In [17]:
# Contar cuántos anuncios tienen EXACTAMENTE precio 9999
print('Anuncios con precio == 9999:', (df['price'] == 9999).sum())

# Ver la distribución de precios > 1000 pero excluyendo el posible placeholder 9999
print(df[(df['price'] > 1000) & (df['price'] != 9999)]['price'].sort_values(ascending=False).head(20))

Anuncios con precio == 9999: 16
274      9856
3792     9856
15956    9785
1443     9356
229      9142
8189     9000
5435     8930
17101    8469
3575     8465
10014    8400
8139     8399
8144     8399
8141     8399
8145     8399
3572     8000
3589     8000
3580     8000
3791     8000
9268     8000
3588     8000
Name: price, dtype: int64


In [18]:
# Ver si hay valores de price que se repiten sospechosamente muchas veces (>1000)
df[df['price'] > 1000]['price'].value_counts().sort_values(ascending=False).head(15)

price
1139    36
1500    20
9999    16
1200    16
2000    13
8000    12
1100    11
3000    10
1400     4
5000     4
8399     4
1700     4
1050     4
6000     3
2500     3
Name: count, dtype: int64

In [19]:
# Investigar los anuncios con price == 1139
df[df['price'] == 1139][['name', 'neighbourhood', 'room_type', 'price', 'minimum_nights']].head(15)

,name,neighbourhood,room_type,price,minimum_nights
1918,"Luxury & Bright, Central Apartment",Palacio,Entire home/apt,1139,1
2174,Silva 2 (Fab),Palacio,Entire home/apt,1139,1
2710,"New & Bright, Central Apartment",Cortes,Entire home/apt,1139,1
2713,"Family & Bright, Central Apartment",Cortes,Entire home/apt,1139,1
3361,"Palace & Bright, Central Apartment",Cortes,Entire home/apt,1139,1
3362,"Golden & Bright, Central Apartment",Cortes,Entire home/apt,1139,1
4361,"Awesome & Bright, Central Apartment",Embajadores,Entire home/apt,1139,1
4665,Preciados 33 (Ama),Sol,Entire home/apt,1139,1
4666,Preciados 33 (Lo),Palacio,Entire home/apt,1139,1
5798,"Cozy & Bright, Central Apartment",Cortes,Entire home/apt,1139,1


In [20]:
# Ver si los anuncios con price == 1139 comparten host_id
df[df['price'] == 1139]['host_id'].value_counts().head(10)

host_id
44267738     25
196632516    11
Name: count, dtype: int64

In [21]:
# Investigar los anuncios con price == 0
df[df['price'] == 0][['name', 'neighbourhood', 'room_type', 'price', 'minimum_nights']]

,name,neighbourhood,room_type,price,minimum_nights
15732,Hotel Santo Domingo,Palacio,Hotel room,0,1
15733,Petit Hostel,Imperial,Hotel room,0,1
15877,TÓTEM Madrid Hotel Boutique,Recoletos,Hotel room,0,1
15912,Scout Madrid Hostel,Media Legua,Hotel room,0,1
15913,Ok Hostel Madrid,Embajadores,Hotel room,0,1
16248,NH Madrid Atocha,Jerónimos,Hotel room,0,1
17462,BLESS Hotel Madrid 5*,Recoletos,Hotel room,0,1
17896,Pestana Plaza Mayor,Sol,Hotel room,0,1


In [22]:
# Reemplazar los valores erróneos (0 y 9999) por nulo, ya que no representan precios reales
df.loc[df['price'].isin([0, 9999]), 'price'] = np.nan

# Comprobar cuántos nulos quedan ahora en price
df['price'].isnull().sum()

np.int64(24)

In [23]:
# Estadísticas descriptivas de minimum_nights
df['minimum_nights'].describe()

count    19618.000000
mean         6.586196
std         33.286582
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max       1125.000000
Name: minimum_nights, dtype: float64

In [24]:
# Ver cuántos anuncios tienen minimum_nights muy alto (más de 365, un año)
print('Anuncios con minimum_nights > 365:', (df['minimum_nights'] > 365).sum())

# Ver los 15 valores más altos
print(df['minimum_nights'].sort_values(ascending=False).head(15))

Anuncios con minimum_nights > 365: 15
10054    1125
4823     1125
12966    1125
11555    1125
10369    1125
9172     1000
7132     1000
10502    1000
12737     999
8189      800
7881      800
2245      750
6194      700
4539      600
9679      500
Name: minimum_nights, dtype: int64


In [25]:
# Investigar los 15 anuncios con minimum_nights > 365
df[df['minimum_nights'] > 365][['name', 'room_type', 'price', 'minimum_nights', 'availability_365']]

,name,room_type,price,minimum_nights,availability_365
2245,Great studio in the main center of Madrid.,Entire home/apt,69.0,750,365
4539,CONFORTABLE HABITACIÓN PARA UNA O DOS PERSONAS,Private room,18.0,600,89
4823,Habitación privada en el centro con wifi y tv,Private room,36.0,1125,0
6194,Apartamento 1 dormitorio Quevedo,Entire home/apt,500.0,700,89
7132,Loft Best neighbourhood in the world! Book!,Entire home/apt,1875.0,1000,327
7881,Habitación individual en Puerta del Angel,Private room,100.0,800,0
8189,habitacion por meses,Private room,9000.0,800,364
9172,Como en casa,Hotel room,89.0,1000,365
9679,Modern Apartment Atocha - 5 bedrooms,Entire home/apt,180.0,500,365
10054,Your home en Madrid,Private room,30.0,1125,0


## Resumen de la limpieza

| Columna | Problema encontrado | Acción |
|---|---|---|
| `latitude` | Texto con formato roto | Corregida a float |
| `longitude` | Formato roto (float) | Corregida |
| `name` | 3 nulos | Rellenado "Sin título" |
| `host_name` | 527 nulos | Rellenado "Desconocido" |
| `reviews_per_month` | 5.637 nulos | Rellenado con 0 |
| `last_review` | 5.637 nulos | Dejado como nulo (info válida) |
| `price` | 24 valores erróneos (0, 9999) | Convertidos a nulo |
| `minimum_nights` | Valores altos | Revisados, legítimos, sin cambios |
| `number_of_reviews`, `calculated_host_listings_count` | — | Limpias, sin cambios |

Con esto queda cerrada la limpieza del dataset. A continuación se guarda la versión procesada.

In [26]:
# Estadísticas descriptivas de number_of_reviews y calculated_host_listings_count
print(df['number_of_reviews'].describe())
print()
print(df['calculated_host_listings_count'].describe())

count    19618.000000
mean        31.858803
std         63.938997
min          0.000000
25%          0.000000
50%          4.000000
75%         31.000000
max        706.000000
Name: number_of_reviews, dtype: float64

count    19618.000000
mean        10.229177
std         23.546472
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        163.000000
Name: calculated_host_listings_count, dtype: float64


In [27]:
# Guardar dataset limpio
df.to_csv('../data/processed/madrid_airbnb_clean.csv', index=False)
print(f"Dataset guardado: {df.shape[0]} filas, {df.shape[1]} columnas")

Dataset guardado: 19618 filas, 16 columnas
